# Modelo de Riesgo Social — 1. Creación de variables

**Objetivo:** predecir, *antes* de despachar una orden, la probabilidad de que el cliente
reaccione de forma agresiva contra la brigada.

Este notebook **no entrena nada**. Solo construye el universo de datos y las variables
(*features*). Es el equivalente al `CreacionVariablesModelos.ipynb` del modelo de propensión
de pago. El entrenamiento va en el notebook `2_Modelo_RiesgoSocial.ipynb`.

---

## Las 3 reglas que rigen todo este notebook

**1. Una fila = una orden gestionada.** No una por cliente. El mismo NIC aparece varias veces,
una por cada visita, y en cada una preguntamos: *¿hubo agresión en esta visita?*

**2. Solo se pueden usar variables conocidas ANTES de la visita.** Esta es la regla más
importante y la más fácil de romper. La tabla `historico_mo` mezcla dos tipos de columnas:

| Se conoce ANTES (se puede usar) | Se conoce DESPUÉS (prohibido usar) |
|---|---|
| `nic`, `deuda_act`, `cant_factura_act` | `accion`, `av_resultado`, `estado_norm` |
| `tipo_os`, `tipo_suspension_solicitada` | `categoria_obs`, `obs_*`, `tl_estandarizado` |
| `tarifa`, `zona`, `municipio`, `barrio` | `subaccion_subanomalia`, `sello_*` |
| `contrata`, `brigada_homologada` | `hora_fin`, `fecha_sincronizacion`, `link_acta` |

Si metes `accion` como variable predictora, el modelo dará 100% de acierto y será **inútil**:
estarías prediciendo el resultado con el resultado. Eso se llama *data leakage* (fuga de datos).
Las columnas de la derecha solo se usan de dos formas: (a) para construir el **target**, y
(b) **agregadas del pasado** del cliente (lo que pasó en visitas *anteriores* sí se conoce hoy).

**3. Todo agregado histórico excluye la fila actual.** Si calculo "tasa de agresión del cliente"
usando también la visita que estoy prediciendo, contamino la variable. Por eso más abajo verás
siempre `cumsum() - fila_actual` y `.shift(1)`.

---
## Hallazgos del diagnóstico de datos

Antes de escribir código revisé la tabla. Cuatro cosas que condicionan el diseño:

**a) El 44% de las órdenes nunca se ejecutó.** Hay 207.969 filas con `accion = 'SIN GESTION'`,
sin técnico asignado y sin `estado_norm`. Son órdenes despachadas que nadie trabajó. Si las dejo
en el universo estoy metiendo casos donde la agresión *era imposible por construcción* y diluyo
la señal. **Se excluyen.** El universo real son 265.049 visitas y la tasa de agresión sube de
2,2% a **3,9%**.

**b) `categoria_obs` no sirve como target.** La categoría `'USUARIO AGRESIVO'` es la etiqueta
más limpia conceptualmente, pero solo existe desde **julio de 2026** (viene de clasificar el texto
de `obs_combinada`, que está vacío en los cortes viejos). Entrenar con ella dejaría 6 meses de
historia fuera. **El target es `accion = 'RESISTENCIA DEL CLIENTE'`**, que está poblada en todo
el periodo. `categoria_obs` se conserva aparte para *validar* el modelo sobre jul–ago.

**c) Hay un artefacto de encoding que parece señal y no lo es.** `descripcion_de_tipo_os` tiene
`'SUSPENSIÓN DEL SERVICIO MD'` (con tilde, 1,25% de agresión) y `'SUSPENSION DEL SERVICIO MD'`
(sin tilde, 6,35%). Es el **mismo tipo de orden**: la diferencia es de qué archivo de corte vino.
Un modelo lo tomaría como predictor potente y en producción fallaría. Por eso normalizamos
acentos y mayúsculas en todas las categóricas. Por lo mismo, la columna `region`
(`SUR_30-03-2026`) **no se usa**: codifica el archivo de origen, no el territorio.

**d) El 43% de las visitas son a clientes sin historial previo.** De 200.182 NICs, 84.277 tienen
una sola visita. Para ellos todas las variables tipo "agresiones previas" van en nulo. Un modelo
basado *solo* en historia del cliente sería ciego para casi la mitad de los casos. Por eso
construimos también **variables de entorno** (tasa de agresión del barrio, del transformador,
del circuito), que sí están disponibles desde la primera visita.

## 1. Carga de librerías y conexión

In [ ]:
%pip install -q pandas numpy matplotlib seaborn python-dotenv sqlalchemy psycopg2-binary pyarrow

In [ ]:
import os
import time
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
from sqlalchemy import create_engine

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 250)

In [ ]:
# --- Rutas del proyecto (todo relativo a esta carpeta) ---
import os

DIR_DATOS   = 'datos'      # insumos y dataset construido
DIR_MODELOS = 'modelos'    # modelo entrenado y su metadata
DIR_SALIDAS = 'salidas'    # rankings y reportes para operaciones

for _d in (DIR_DATOS, DIR_MODELOS, DIR_SALIDAS):
    os.makedirs(_d, exist_ok=True)

In [ ]:
load_dotenv()

SCR_DATABASE_URL = os.getenv("SCR_DATABASE_URL")
engine = create_engine(SCR_DATABASE_URL)

### 1.1 Descarga de `historico_mo`

Pedimos solo las columnas que vamos a usar (la tabla tiene 71 y varias son texto libre pesado).
La descarga tarda **~7 minutos**, así que la guardamos en un archivo local `parquet`: la próxima
vez que abras el notebook lee del disco en segundos. Borra el `.parquet` cuando quieras refrescar.

In [ ]:
CACHE = f'{DIR_DATOS}/historico_mo_raw.parquet'

COLUMNAS = """orden, nic, fecha_cierre, tipo_os, descripcion_de_tipo_os, tipo_suspension_solicitada,
tipo_brigada, brigada_homologada, contrata, tecnico, territorio, zona, municipio, localidad_barrio,
corregimiento, tarifa, direccion, id_transformador, id_circuito, deuda_act, deuda_cierre,
cant_factura_act, cant_factura_cierre, valor_orden, accion, av_resultado, estado_norm,
categoria_obs, caracterizacion_del_predio, hora_inicio"""

if os.path.exists(CACHE):
    df_mo = pd.read_parquet(CACHE)
    print('leido de cache local')
else:
    t0 = time.time()
    df_mo = pd.read_sql(
        f"SELECT {COLUMNAS} FROM dbanalitica.historico_mo WHERE fecha_cierre IS NOT NULL",
        engine
    )
    df_mo.to_parquet(CACHE, index=False)
    print(f'descargado de la BD en {time.time()-t0:.0f}s')

print(df_mo.shape)
df_mo.head(3)

## 2. Normalización de texto

Quitamos tildes, pasamos a mayúsculas y colapsamos espacios dobles. Esto arregla el problema (c)
del diagnóstico: sin esto, la misma categoría escrita de dos formas se convierte en dos variables
distintas y el modelo aprende el archivo de origen en vez del negocio.

In [ ]:
def norm_txt(s):
    """Quita tildes, pasa a MAYUSCULAS y colapsa espacios. NaN se mantiene como NaN."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    return " ".join(s.upper().split())


COLS_TEXTO = [
    'accion', 'descripcion_de_tipo_os', 'tipo_suspension_solicitada', 'categoria_obs',
    'municipio', 'localidad_barrio', 'zona', 'territorio', 'tarifa', 'brigada_homologada',
    'tipo_brigada', 'contrata', 'estado_norm', 'caracterizacion_del_predio',
]

for c in COLS_TEXTO:
    df_mo[c] = df_mo[c].map(norm_txt)

In [ ]:
# Comprobacion del hallazgo (c): ahora cada tipo_os tiene UNA sola descripcion
df_mo.groupby(['tipo_os', 'descripcion_de_tipo_os']).size()

## 3. Definición del universo

Nos quedamos con las órdenes **efectivamente gestionadas**. Una orden `SIN GESTION` es una visita
que no ocurrió: no puede haber agresión, y mezclarla con las reales solo agrega ruido.

Después ordenamos por `nic` y `fecha_cierre`. **Ese orden es obligatorio**: todas las variables
históricas de la sección 5 dependen de que las visitas de cada cliente estén en secuencia
cronológica. Incluimos `orden` como tercer criterio para que el resultado sea determinista cuando
un mismo NIC tiene dos visitas el mismo día (pasa en ~17.400 casos).

In [ ]:
antes = len(df_mo)

df = df_mo[df_mo['accion'] != 'SIN GESTION'].copy()

print(f"Ordenes totales:     {antes:>8,}")
print(f"Sin gestion (fuera): {antes - len(df):>8,}")
print(f"Universo de trabajo: {len(df):>8,}")

In [ ]:
df['fecha_cierre'] = pd.to_datetime(df['fecha_cierre'])

df = (
    df.sort_values(['nic', 'fecha_cierre', 'orden'])
      .reset_index(drop=True)
)

print('rango de fechas:', df['fecha_cierre'].min().date(), '->', df['fecha_cierre'].max().date())
print('NICs distintos:', df['nic'].nunique())

## 4. El target (la variable a predecir)

Definimos tres versiones. **La principal es `AGRESIVO`**; las otras dos sirven para análisis de
sensibilidad, es decir, para comprobar después que las conclusiones no dependen de una decisión
arbitraria de etiquetado.

| Variable | Definición | Uso |
|---|---|---|
| `AGRESIVO` | `accion = 'RESISTENCIA DEL CLIENTE'` | **Target del modelo.** Cubre todo el periodo |
| `AGRESIVO_AMPLIO` | resistencia **o** acceso impedido / difícil acceso | Hostilidad + obstrucción pasiva |
| `AGRESIVO_TEXTO` | `categoria_obs = 'USUARIO AGRESIVO'` | Solo jul–ago. Para **validar**, no para entrenar |

Cambia `TARGET` en la celda de abajo si el negocio decide que la definición correcta es la amplia;
todo el resto del notebook se recalcula solo.

In [ ]:
TARGET = 'AGRESIVO'   # <- cambia aqui si quieres entrenar con la definicion amplia

df['AGRESIVO'] = (df['accion'] == 'RESISTENCIA DEL CLIENTE').astype(int)

df['AGRESIVO_AMPLIO'] = df['accion'].isin(
    ['RESISTENCIA DEL CLIENTE', 'ACCESO IMPEDIDO', 'DIFICIL ACCESO']
).astype(int)

df['AGRESIVO_TEXTO'] = (df['categoria_obs'] == 'USUARIO AGRESIVO').astype(int)

In [ ]:
resumen = pd.DataFrame({
    'casos': df[['AGRESIVO', 'AGRESIVO_AMPLIO', 'AGRESIVO_TEXTO']].sum(),
    'tasa_%': (df[['AGRESIVO', 'AGRESIVO_AMPLIO', 'AGRESIVO_TEXTO']].mean() * 100).round(2),
})
resumen

### 4.1 Desbalance de clases

Solo ~4% de las visitas terminan en agresión. Esto es un problema **desbalanceado** y tiene dos
consecuencias que hay que tener presentes en el notebook 2:

- La *accuracy* es una métrica engañosa: un modelo que responda "nunca hay agresión" acierta el
  96% de las veces y no sirve para nada. Se evalúa con **AUC-PR, recall y precisión**.
- Lo natural sería compensar con `class_weights`, pero **probamos y resulta contraproducente**:
  no mejora el ordenamiento y arruina la calibración (Brier 0,17 vs 0,04). Ver notebook 2, sección 3.

4% es un desbalance perfectamente manejable — mucho mejor que el típico caso de fraude (0,1%).

In [ ]:
# Evolucion mensual del target: sirve para detectar cambios de criterio de registro
tmp = df.copy()
tmp['PERIODO'] = tmp['fecha_cierre'].dt.to_period('M').astype(str)

evol = tmp.groupby('PERIODO')['AGRESIVO'].agg(visitas='size', tasa='mean')
evol['tasa'] = (evol['tasa'] * 100).round(2)
evol

> **Ojo con esto.** La tasa sube de 3,2% en enero a 5,4% en agosto. Puede ser un aumento real de
> conflictividad, o puede ser que se empezó a registrar mejor. No lo podemos distinguir con los
> datos que tenemos. Impacto práctico: el modelo entrenado con meses viejos subestimará el riesgo
> en meses nuevos. Se corrige recalibrando cada cierto tiempo, y es una razón más para hacer el
> split train/test **por tiempo** y no al azar.

## 5. Variables de contexto de la orden

Lo que se sabe en el momento de despachar, sin mirar el historial del cliente.

In [ ]:
# --- Cliente / tarifa: separamos segmento y estrato ---
tar = df['tarifa'].fillna('SIN DATO').str.split('|', n=1, expand=True)

df['SEGMENTO'] = tar[0].str.strip()
df['ESTRATO'] = pd.to_numeric(tar[1].str.extract(r'(\d)')[0], errors='coerce')

df[['tarifa', 'SEGMENTO', 'ESTRATO']].drop_duplicates().head(10)

In [ ]:
# --- Variables temporales ---
df['ANIO'] = df['fecha_cierre'].dt.year
df['MES'] = df['fecha_cierre'].dt.month
df['DIA_MES'] = df['fecha_cierre'].dt.day
df['DIA_SEMANA'] = df['fecha_cierre'].dt.dayofweek        # 0=lunes
df['ES_FIN_SEMANA'] = (df['DIA_SEMANA'] >= 5).astype(int)
df['PERIODO'] = df['fecha_cierre'].dt.to_period('M').astype(str)

# Codificacion ciclica del mes: diciembre (12) y enero (1) quedan cerca entre si,
# cosa que el numero 12 vs 1 no logra transmitir. Mismo truco que uso Karen.
df['MES_SIN'] = np.sin(2 * np.pi * df['MES'] / 12)
df['MES_COS'] = np.cos(2 * np.pi * df['MES'] / 12)

# Hora de la visita (la de INICIO se conoce al llegar; hora_fin seria leakage)
df['HORA_VISITA'] = (
    pd.to_timedelta(df['hora_inicio'].astype(str), errors='coerce').dt.total_seconds() / 3600
)

In [ ]:
# --- Variables economicas de la orden ---
df['DEUDA_POR_FACTURA'] = np.where(
    df['cant_factura_act'].fillna(0) > 0,
    df['deuda_act'] / df['cant_factura_act'],
    np.nan
)

# log1p comprime la escala: la deuda va de miles a cientos de millones y unos
# pocos clientes enormes dominarian la variable sin esta transformacion.
df['LOG_DEUDA'] = np.log1p(df['deuda_act'].clip(lower=0))

## 6. Variables históricas del cliente

El corazón del modelo: *"en base a su histórico"*.

**La técnica clave.** Para cada fila necesito los acumulados del cliente **sin incluir esa misma
fila**. El patrón es siempre el mismo:

```python
acumulado_del_grupo  -  valor_de_la_fila_actual   # = solo el pasado
```

Y para los lags, `.shift(1)` dentro del `groupby`. Si se te olvida el `- df[col]` o el `.shift(1)`,
el modelo se ve espectacular en el test y luego fracasa en producción. Al final de esta sección
hay chequeos que verifican que no pasó.

In [ ]:
def acumulado_previo(frame, key, col):
    """Suma acumulada de `col` dentro de cada `key`, EXCLUYENDO la fila actual.

    Es el ladrillo con el que se construyen todas las variables historicas:
    garantiza que cada fila solo ve su propio pasado.
    """
    return frame.groupby(key, sort=False)[col].cumsum() - frame[col]

In [ ]:
g = df.groupby('nic', sort=False)

# Cuantas visitas habia tenido este cliente ANTES de esta
df['VISITAS_PREVIAS'] = g.cumcount()
df['ES_PRIMERA_VISITA'] = (df['VISITAS_PREVIAS'] == 0).astype(int)

# Historial de agresion
df['AGRESIONES_PREVIAS'] = acumulado_previo(df, 'nic', 'AGRESIVO')

df['TASA_AGRESION_HIST'] = np.where(
    df['VISITAS_PREVIAS'] > 0,
    df['AGRESIONES_PREVIAS'] / df['VISITAS_PREVIAS'],
    np.nan                      # sin historia -> NaN, NO cero (cero seria mentir)
)

### 6.1 Otros comportamientos previos

La agresión no es el único rastro de un cliente conflictivo. Marcamos cinco conductas más y de
cada una calculamos el acumulado previo. Estas columnas auxiliares empiezan con `_` porque se
derivan de `accion` (post-visita) y **se borran antes de guardar**: solo sobrevive su versión
histórica, que sí es legítima.

La más interesante es `RECONEX_NO_AUTORIZ_PREVIAS`: un cliente que se auto-reconecta después de
que le suspenden ya demostró disposición a confrontar a la empresa.

In [ ]:
df['_FALLIDA'] = df['estado_norm'].isin(['FALLIDA', 'PERDIDA']).astype(int)

df['_ACCESO_IMPEDIDO'] = df['accion'].isin(['ACCESO IMPEDIDO', 'DIFICIL ACCESO']).astype(int)

df['_PAGO_EVITO'] = (df['accion'] == 'CLIENTE HA CANCELADO (PAGO RECIENTE)').astype(int)

df['_RECONEX_NO_AUTORIZ'] = df['accion'].str.contains(
    'RECONEXION NO AUTORIZADA|AUTORECONECTADO', case=False, na=False
).astype(int)

df['_SUSP_EFECTIVA'] = df['accion'].str.startswith('EXITO - SUSPENSION', na=False).astype(int)

In [ ]:
DERIVADAS = [
    ('_FALLIDA',            'FALLIDAS_PREVIAS'),
    ('_ACCESO_IMPEDIDO',    'ACCESO_IMPEDIDO_PREVIAS'),
    ('_PAGO_EVITO',         'PAGOS_EVITARON_PREVIAS'),
    ('_RECONEX_NO_AUTORIZ', 'RECONEX_NO_AUTORIZ_PREVIAS'),
    ('_SUSP_EFECTIVA',      'SUSP_EFECTIVAS_PREVIAS'),
]

for origen, destino in DERIVADAS:
    df[destino] = acumulado_previo(df, 'nic', origen)

df['TASA_FALLA_HIST'] = np.where(
    df['VISITAS_PREVIAS'] > 0,
    df['FALLIDAS_PREVIAS'] / df['VISITAS_PREVIAS'],
    np.nan
)

### 6.2 Lags y ventanas móviles

`TASA_AGRESION_HIST` promedia toda la vida del cliente y por eso reacciona lento. Estas variables
capturan lo **reciente**, que suele pesar más: un cliente que fue agresivo la visita pasada es
distinto de uno que lo fue hace un año.

In [ ]:
# Que paso en la visita inmediatamente anterior
df['AGRESIVO_V1'] = g['AGRESIVO'].shift(1)

# Agresiones en las ultimas 3 y 5 visitas (shift(1) = sin contar la actual)
for k in (3, 5):
    df[f'AGRESIONES_ULT_{k}V'] = g['AGRESIVO'].transform(
        lambda x: x.shift(1).rolling(k, min_periods=1).sum()
    )

df['FALLIDAS_ULT_3V'] = g['_FALLIDA'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).sum()
)

In [ ]:
# --- Distancia temporal ---
df['DIAS_DESDE_ULT_VISITA'] = (df['fecha_cierre'] - g['fecha_cierre'].shift(1)).dt.days

# Dias desde la ultima agresion: tomamos la fecha solo cuando hubo agresion,
# la desplazamos una posicion (para no ver la actual) y la arrastramos con ffill.
fecha_si_agresion = df['fecha_cierre'].where(df['AGRESIVO'] == 1)

ult_agresion = (
    fecha_si_agresion
    .groupby(df['nic'], sort=False).shift(1)
    .groupby(df['nic'], sort=False).ffill()
)

df['DIAS_DESDE_ULT_AGRESION'] = (df['fecha_cierre'] - ult_agresion).dt.days

In [ ]:
# --- Racha de visitas consecutivas sin agresion ---
# Cada agresion previa abre un "bloque" nuevo; la racha es la posicion dentro del bloque.
agresion_previa = g['AGRESIVO'].shift(1).fillna(0)
bloque = agresion_previa.groupby(df['nic'], sort=False).cumsum()

df['RACHA_SIN_AGRESION'] = df.groupby(['nic', bloque], sort=False).cumcount()
df.loc[df['ES_PRIMERA_VISITA'] == 1, 'RACHA_SIN_AGRESION'] = np.nan

In [ ]:
# --- Evolucion de la deuda entre visitas ---
df['DEUDA_PREV'] = g['deuda_act'].shift(1)
df['DELTA_DEUDA'] = df['deuda_act'] - df['DEUDA_PREV']
df['CRECE_DEUDA'] = (df['DELTA_DEUDA'] > 0).astype(float).where(df['DEUDA_PREV'].notna())

## 7. Variables de entorno (barrio, transformador, circuito)

Aquí resolvemos el hallazgo (d): **el 43% de las visitas son a clientes sin historial**. Para
ellos todo lo de la sección 6 va en nulo, pero el entorno sí se conoce.

Tres detalles de implementación importantes:

**Reordenamos por fecha global.** Las secciones anteriores iban ordenadas por NIC. Un acumulado
de barrio tiene que recorrer el tiempo *calendario*, no el orden por cliente, o mezclaríamos
visitas futuras de un vecino con el pasado de otro.

**Suavizado bayesiano.** La tasa cruda de un barrio con 2 visitas es 0% o 50%, puro ruido. La
fórmula `(agresiones + K·prior) / (visitas + K)` arranca en el promedio general y se va moviendo
hacia el dato real a medida que el barrio acumula visitas. Con `K=20`, un barrio necesita ~20
visitas para que su tasa propia pese la mitad.

**`TASA_AGRESION_TECNICO` no es una variable del cliente.** La calculamos porque es un
diagnóstico valioso — si un técnico reporta el triple de resistencia que sus compañeros en zonas
comparables, eso dice algo del técnico, no de los clientes. **No la metas como feature en el
notebook 2**: el modelo debe predecir el riesgo del cliente, no adivinar a quién le tocó ir.

In [ ]:
PRIOR = df['AGRESIVO'].mean()   # tasa global de agresion
K = 20                          # fuerza del suavizado


def tasa_entorno(frame, key, nombre, target='AGRESIVO', k=K):
    """Tasa historica de agresion del grupo `key`, suavizada y sin la fila actual."""
    n_previas = frame.groupby(key, sort=False).cumcount()
    pos_previas = acumulado_previo(frame, key, target)

    frame[f'VISITAS_PREV_{nombre}'] = n_previas
    frame[f'TASA_AGRESION_{nombre}'] = (pos_previas + k * PRIOR) / (n_previas + k)

In [ ]:
# IMPORTANTE: reordenar por tiempo global antes de acumular por entorno
df = df.sort_values(['fecha_cierre', 'orden']).reset_index(drop=True)

ENTORNOS = [
    ('localidad_barrio', 'BARRIO'),
    ('id_transformador', 'TRAFO'),
    ('id_circuito',      'CIRCUITO'),
    ('municipio',        'MUNICIPIO'),
]

for columna, nombre in ENTORNOS:
    tasa_entorno(df, columna, nombre)

# Diagnostico operativo, NO feature del modelo
tasa_entorno(df, 'tecnico', 'TECNICO')

## 8. Validación: ¿las variables están bien construidas?

Nunca pases al entrenamiento sin correr estos chequeos. Los tres primeros detectan fugas de datos;
el cuarto muestra si las variables realmente discriminan.

In [ ]:
# CHEQUEO 1 - coherencia logica: no puedo tener mas agresiones que visitas
ok1 = (df['AGRESIONES_PREVIAS'] <= df['VISITAS_PREVIAS']).all()

# CHEQUEO 2 - la primera visita de un cliente no puede tener historia
primera = df['ES_PRIMERA_VISITA'] == 1
ok2 = (df.loc[primera, 'AGRESIONES_PREVIAS'].max() == 0
       and df.loc[primera, 'TASA_AGRESION_HIST'].isna().all())

# CHEQUEO 3 - la variable historica NO puede reproducir el target exactamente
ok3 = (df['AGRESIVO'] != df['AGRESIONES_PREVIAS']).any()

print('1. agresiones_previas <= visitas_previas :', ok1)
print('2. primera visita sin historia           :', ok2)
print('3. sin identidad con el target           :', ok3)
assert ok1 and ok2 and ok3, 'Revisa la construccion: hay fuga de datos'

In [ ]:
# CHEQUEO 4 - poder predictivo bruto de las variables clave.
# Si una variable no separa aqui, dificilmente aportara en el modelo.

print('--- Segun lo ocurrido en la visita anterior ---')
print(df.groupby(df['AGRESIVO_V1'].fillna(-1))['AGRESIVO']
        .agg(tasa='mean', n='size').assign(tasa=lambda d: (d.tasa*100).round(2)))

print('\n--- Segun agresiones previas acumuladas ---')
print(df.groupby(pd.cut(df['AGRESIONES_PREVIAS'], [-1, 0, 1, 2, 100]), observed=True)['AGRESIVO']
        .agg(tasa='mean', n='size').assign(tasa=lambda d: (d.tasa*100).round(2)))

print('\n--- Segun quintil de riesgo del barrio ---')
print(df.groupby(pd.qcut(df['TASA_AGRESION_BARRIO'], 5, duplicates='drop'), observed=True)['AGRESIVO']
        .agg(tasa='mean', n='size').assign(tasa=lambda d: (d.tasa*100).round(2)))

**Cómo leer el chequeo 4.** Si el cliente fue agresivo en la visita anterior, la tasa pasa de
2,8% a **30%** (11 veces más). Con 3 o más agresiones acumuladas llega al **50%**. Y el barrio
solo ya separa de 0,9% a 7,3% entre el quintil más tranquilo y el más conflictivo — cubriendo el
100% de las filas, incluidos los clientes nuevos.

Traducción: **sí hay señal y el problema es predecible.** Ya podemos entrenar con confianza.

Falta un chequeo más —el que verifica que ninguna variable sea el resultado disfrazado—, pero necesita la lista de features: va en la 9.1.

In [ ]:
# Nulos de las variables historicas: se espera ~43% en las del cliente (los que no tienen
# historia) y 0% en las de entorno. CatBoost maneja los NaN nativamente, no hay que imputar.
HIST = ['TASA_AGRESION_HIST', 'AGRESIVO_V1', 'DIAS_DESDE_ULT_VISITA', 'DIAS_DESDE_ULT_AGRESION',
        'RACHA_SIN_AGRESION', 'DELTA_DEUDA', 'TASA_AGRESION_BARRIO', 'TASA_AGRESION_TRAFO']

(df[HIST].isna().mean() * 100).round(1).rename('% nulos').to_frame()

## 9. Organización y guardado

Separamos explícitamente qué columna es qué. Esta lista es el contrato con el notebook 2: lo que
esté en `PROHIBIDAS` no puede entrar al modelo bajo ningún concepto.

In [ ]:
IDENTIFICADORES = ['orden', 'nic', 'fecha_cierre', 'PERIODO']

CATEGORICAS = [
    'tipo_os', 'tipo_suspension_solicitada', 'SEGMENTO',
    'zona', 'territorio', 'municipio', 'localidad_barrio',
    'contrata', 'brigada_homologada', 'tipo_brigada',
]

NUMERICAS = [
    # contexto de la orden
    'ESTRATO', 'deuda_act', 'deuda_cierre', 'cant_factura_act', 'cant_factura_cierre',
    'DEUDA_POR_FACTURA', 'LOG_DEUDA',   # ojo: valor_orden NO va (ver 8.1)
    # tiempo
    'MES', 'DIA_MES', 'DIA_SEMANA', 'ES_FIN_SEMANA', 'MES_SIN', 'MES_COS', 'HORA_VISITA',
    # historia del cliente
    'VISITAS_PREVIAS', 'ES_PRIMERA_VISITA', 'AGRESIONES_PREVIAS', 'TASA_AGRESION_HIST',
    'AGRESIVO_V1', 'AGRESIONES_ULT_3V', 'AGRESIONES_ULT_5V',
    'FALLIDAS_PREVIAS', 'TASA_FALLA_HIST', 'FALLIDAS_ULT_3V',
    'ACCESO_IMPEDIDO_PREVIAS', 'PAGOS_EVITARON_PREVIAS',
    'RECONEX_NO_AUTORIZ_PREVIAS', 'SUSP_EFECTIVAS_PREVIAS',
    'DIAS_DESDE_ULT_VISITA', 'DIAS_DESDE_ULT_AGRESION', 'RACHA_SIN_AGRESION',
    'DELTA_DEUDA', 'CRECE_DEUDA',
    # entorno
    'TASA_AGRESION_BARRIO', 'VISITAS_PREV_BARRIO',
    'TASA_AGRESION_TRAFO', 'VISITAS_PREV_TRAFO',
    'TASA_AGRESION_CIRCUITO', 'VISITAS_PREV_CIRCUITO',
    'TASA_AGRESION_MUNICIPIO',
]

TARGETS = ['AGRESIVO', 'AGRESIVO_AMPLIO', 'AGRESIVO_TEXTO']

# Se conocen DESPUES de la visita. Se guardan para analizar, NUNCA como features.
PROHIBIDAS = ['accion', 'av_resultado', 'estado_norm', 'categoria_obs',
              'valor_orden',   # se liquida SEGUN el resultado -> fuga (ver 8.1)
              'TASA_AGRESION_TECNICO', 'VISITAS_PREV_TECNICO', 'tecnico']

FEATURES = CATEGORICAS + NUMERICAS
print(f'{len(FEATURES)} features: {len(CATEGORICAS)} categoricas + {len(NUMERICAS)} numericas')

### 9.1 Chequeo 5 — barrido de fugas (el que salvó este modelo)

Los chequeos 1-3 verifican la construcción; el 4 mira si hay señal. Falta el más importante:
**revisar variable por variable que ninguna sea, en realidad, el resultado disfrazado.**

El barrido calcula el AUC de cada feature *por sí sola*. Interpretación:

- **AUC ~0,5** → la variable no aporta nada
- **AUC 0,60–0,75** → aporte normal y sano
- **AUC > 0,90** → alarma: una sola variable no puede predecir casi perfecto. Es fuga

En la primera corrida esto detectó `valor_orden` con **AUC 0,988**. Al mirarlo de cerca: las
10.400 órdenes con agresión tienen **todas** `valor_orden = 0`, y ninguna orden con valor > 0
terminó en agresión. La razón es de negocio — el valor se liquida *según el resultado*: si la
visita se pierde por resistencia, al contratista no se le paga. Es una columna que solo se llena
**después** de la visita.

Un modelo con esa variable habría dado ~99% de acierto en pruebas y **cero utilidad en producción**,
porque al momento de despachar la orden ese valor todavía no existe. Por eso `valor_orden` está
ahora en `PROHIBIDAS`.

**Corre siempre este chequeo antes de entrenar**, sobre todo si agregas columnas nuevas.

In [ ]:
from sklearn.metrics import roc_auc_score

def barrido_fugas(frame, features, target='AGRESIVO'):
    """AUC de cada feature por si sola. >0.90 = casi seguro fuga de datos."""
    y = frame[target]
    filas = []
    for f in features:
        s = frame[f]
        if pd.api.types.is_numeric_dtype(s):
            score = roc_auc_score(y, s.fillna(s.median()))
        else:
            # para categoricas: la tasa media del grupo como score
            score = roc_auc_score(y, y.groupby(s.fillna('NA')).transform('mean'))
        filas.append((f, max(score, 1 - score)))

    r = (pd.DataFrame(filas, columns=['feature', 'auc'])
           .sort_values('auc', ascending=False).reset_index(drop=True))
    r['alerta'] = np.where(r['auc'] > 0.90, 'FUGA?', '')
    return r


barrido = barrido_fugas(df, FEATURES)
print(barrido.head(12).round(4).to_string(index=False))

sospechosas = barrido.loc[barrido['auc'] > 0.90, 'feature'].tolist()
assert not sospechosas, f'Revisa estas variables antes de seguir: {sospechosas}'
print('\nSin fugas detectadas.')

In [ ]:
# Las columnas auxiliares con "_" se derivan de la accion de la visita actual: fuera.
df = df.drop(columns=[c for c in df.columns if c.startswith('_')])

df_universo = df[IDENTIFICADORES + FEATURES + TARGETS + PROHIBIDAS].copy()

print(df_universo.shape)
df_universo.head(3)

In [ ]:
df_universo.to_parquet(f'{DIR_DATOS}/df_universo_agresividad.parquet', index=False)

# CSV para quien quiera abrirlo en Excel (mas pesado y mas lento de leer)
df_universo.to_csv(f'{DIR_DATOS}/df_universo_agresividad.csv', index=False,
                   encoding='utf-8-sig')

import json
with open(f'{DIR_DATOS}/features_agresividad.json', 'w', encoding='utf-8') as f:
    json.dump({'features': FEATURES, 'categoricas': CATEGORICAS,
               'numericas': NUMERICAS, 'target': TARGET,
               'prohibidas': PROHIBIDAS}, f, indent=2, ensure_ascii=False)

print('guardado:', df_universo.shape)

---
## 10. Qué sigue (notebook 2)

1. **Split train/test por tiempo, no al azar.** Entrenar con ene–jun y probar con jul–ago.
   Karen usó `GroupShuffleSplit` por cuenta, que evita que el mismo cliente esté en train y test,
   pero permite entrenar con el futuro y probar con el pasado. Con variables acumulativas como las
   nuestras eso infla el resultado. Lo honesto —y lo que reproduce el uso real— es cortar por fecha.
2. **CatBoost** sin `class_weights` (probado: empeora la calibración), con `cat_features` = `CATEGORICAS`.
3. **Métricas: AUC-PR, recall y precisión**, no accuracy (ver sección 4.1).
4. **SHAP** para explicar qué pesa en cada predicción — imprescindible si la salida va a usarse
   para decidir qué brigada entra a una dirección.
5. **Elegir el umbral con criterio operativo**: no el 0,5 por defecto, sino el que produzca una
   cantidad de alertas que la operación pueda atender.

### Limitaciones que hay que decir de frente

- **Solo hay 7,5 meses de historia** (ene–ago 2026). No se puede capturar estacionalidad anual y
  el "histórico" de cada cliente es corto: la mediana es de 2 visitas.
- **La tasa de agresión sube mes a mes** y no sabemos si es realidad o mejor registro (sección 4.1).
- **El target es lo que el técnico reportó**, no la agresión objetiva. Si un técnico no lo registra,
  para el modelo no ocurrió. Por eso vale la pena mirar `TASA_AGRESION_TECNICO` antes de confiar
  ciegamente en el resultado.
- **43% de las visitas no tienen historia del cliente.** Para ellas el modelo se apoya casi solo en
  entorno y contexto: revisa el desempeño por separado en ese subgrupo.